In [ ]:
# Standard library
import re
from collections import OrderedDict
from functools import partial
import joblib
import lightgbm as gbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
import xgboost as xgb
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, make_scorer, precision_recall_curve, roc_auc_score, roc_curve, auc as sk_auc
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from skopt import BayesSearchCV
from skopt.callbacks import DeadlineStopper
from skopt.space import Categorical, Integer, Real
from statsmodels.stats.outliers_influence import variance_inflation_factor
import skorch
from skorch.callbacks import Callback, EarlyStopping, GradientNormClipping
from skorch.dataset import ValidSplit

In [ ]:

IBAN_COL   = 'Account'
TS_COL     = 'Timestamp'
LABEL_COL  = 'proxy_label' 
HOLDING_PSP_COL = 'To Bank'
DECLARING_PSP_COL = 'From Bank'
N          = 1
train_df = pd.read_parquet('train_df.parquet')
test_df = pd.read_parquet('test_df.parquet')

data = joblib.load('preprocessed.joblib')

X_train_np = data['X_train_np']
y_train = data['y_train']
X_test_np = data['X_test_np']
y_test = data['y_test']
imputer = data['imputer']
scaler = data['scaler']
train_categories = data['train_categories']
all_nan_cols = data['all_nan_cols']
partial_cols = data['partial_cols']
num_lag_cols = data['num_lag_cols']
cat_lag_cols = data['cat_lag_cols']
cat_cardinalities = data['cat_cardinalities']
cat_features = data['cat_features']

y_train = y_train.astype(np.int64)
y_test = y_test.astype(np.int64)



In [ ]:

drop_cols = [LABEL_COL, IBAN_COL, TS_COL,'key_lag1','Unnamed: 0_lag1','fold_lag1','Amount Received_lag1', 'Amount Paid_lag1']

X_test_shap = pd.DataFrame(X_test_np, columns=test_df.drop(columns=drop_cols).columns)
X_train_shap = pd.DataFrame(X_train_np, columns=test_df.drop(columns=drop_cols).columns)

print(X_train_shap[num_lag_cols].describe().loc[['mean','std']])

In [ ]:
# scoring = make_scorer(average_precision_score, response_method="predict_proba")

# deadline = DeadlineStopper(total_time=60*60)  

# sample_weights = compute_sample_weight(
#     class_weight='balanced',
#     y=y_train 
# )

# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# model = xgb.XGBClassifier(
#     enable_categorical=True,
#     tree_method='hist',
#     eval_metric='aucpr', 
#     random_state=42
# )

# search_spaces = {
#     'learning_rate': Real(0.01, 0.5, prior='log-uniform'),
#     'max_depth': Integer(2, 10), #max depth
#     'min_child_weight': Integer(1, 5),  # lowest better for minority classes 
#     'subsample': Real(0.5, 1.0),
#     'colsample_bytree': Real(0.3, 1.0), # Fraction of total training data used to build each tree
#     'reg_lambda': Real(1e-9, 100., prior='log-uniform'),
#     'reg_alpha': Real(1e-9, 100., prior='log-uniform'),
#     'n_estimators': Integer(50, 2000), #Number of trees 
#     'max_delta_step': Integer(3,7), # maximun weight assigned to each tree leaf good for imbalanced classes and regression 
# }

# opt = BayesSearchCV(
#     estimator=model,
#     search_spaces=search_spaces,
#     scoring=scoring,
#     n_iter=100,
#     cv=cv,
#     n_jobs=-1,
#     verbose=2,
#     random_state=42,
#     refit=True,
# ) # Run a BO search for the hyperparameters to find the ebst configuration for the dataset 

# opt.fit(X_train_np, y_train, **{'sample_weight': sample_weights}, callback=deadline)

# print("Best score:", opt.best_score_)
# print("Best params:", opt.best_params_)

In [ ]:

# model = opt.best_estimator_
# preds = model.predict(X_test_np)
# proba = model.predict_proba(X_test_np)
# print(classification_report(y_test, preds))
# auc = roc_auc_score(y_test, proba[:, 1])
# print(f"ROC-AUC: {auc:.4f}")
# joblib.dump(model,r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_xgb_v1.plk')

In [ ]:
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, auc

# Get positive-class probabilities
# proba = model.predict_proba(X_test_np)[:, 1]

# Compute ROC points and AUC
# fpr, tpr, thresholds = roc_curve(y_test, proba)
# roc_auc = auc(fpr, tpr)

# Plot
# plt.figure(figsize=(7, 6))
# plt.plot(fpr, tpr, color='darkorange', lw=2,
#         label=f'XGBoost (AUC = {roc_auc:.4f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')

# Axis formatting
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.legend(loc='lower right')
# plt.grid(alpha=0.3)
# plt.show()

In [ ]:

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # Cross validation

# model = RandomForestClassifier(
#     random_state=42
# )

# sample_weights = compute_sample_weight(
#     class_weight='balanced',
#     y=y_train 
# )

# search_spaces = {
#     'n_estimators': Integer(50, 2000), # Number of trees 
#     'max_depth': Integer(2, 30), #max tree depth
#     'min_samples_leaf': Integer(1, 10),  # lower this to find minorities
#     'class_weight': Categorical(['balanced', 'balanced_subsample']), # Test best results for the macro average 
#     'max_features':      Categorical(['sqrt', 'log2']),
#     'bootstrap':         Categorical([True, False]),
# }

# scoring = make_scorer(average_precision_score,
#                       response_method="predict_proba")

# opt = BayesSearchCV(
#     estimator=model,
#     search_spaces=search_spaces,
#     scoring=scoring,
#     n_iter=100,
#     cv=cv,
#     n_jobs=-1,
#     verbose=2,
#     random_state=42,
#     refit=True,
# ) # Run a BO search for the hyperparameters to find the ebst configuration for the dataset 

# # Stop after max_seconds regardless of n_iter
# deadline = DeadlineStopper(total_time=60*60)  

# opt.fit(X_train_np, y_train, callback=deadline, sample_weight=sample_weights)

# print("Best score:", opt.best_score_)
# print("Best params:", opt.best_params_)

In [ ]:

# model = opt.best_estimator_
# preds = model.predict(X_test_np)
# proba = model.predict_proba(X_test_np)
# print(classification_report(y_test, preds))
# auc = roc_auc_score(y_test, proba[:, 1])
# print(f"ROC-AUC: {auc:.4f}")
# joblib.dump(model,r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_RF_v1.plk')

In [ ]:
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, auc

# # Get positive-class probabilities
# proba = model.predict_proba(X_test_np)[:, 1]

# # Compute ROC points and AUC
# fpr, tpr, thresholds = roc_curve(y_test, proba)
# roc_auc = auc(fpr, tpr)

# # Plot
# plt.figure(figsize=(7, 6))
# plt.plot(fpr, tpr, color='darkorange', lw=2,
#          label=f'Random forest (AUC = {roc_auc:.4f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')

# # Axis formatting
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.legend(loc='lower right')
# plt.grid(alpha=0.3)
# plt.show()

In [ ]:
# scoring = make_scorer(average_precision_score,
#                       response_method="predict_proba",
#                       average='macro')

# deadline = DeadlineStopper(total_time=60*60)  

# sample_weights = compute_sample_weight(
#     class_weight='balanced',
#     y=y_train 
# )

# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# model= gbm.LGBMClassifier(
#     objective='binary',
#     random_state=42,
#     n_jobs=-1
# )

# search_spaces = {
#     'num_leaves': Integer(15, 1000),
#     'max_depth': Integer(3, 12),
#     'learning_rate': Real(1e-3, 0.3, prior='log-uniform'),
#     'n_estimators': Integer(50, 1000),
#     'min_child_samples': Integer(5, 100),
#     'subsample': Real(0.5, 1.0),
#     'colsample_bytree': Real(0.5, 1.0),
#     'reg_alpha': Real(1e-8, 10.0, prior='log-uniform'),
#     'reg_lambda': Real(1e-8, 10.0, prior='log-uniform'),
# }
# opt = BayesSearchCV(
#     estimator=model,
#     search_spaces=search_spaces,
#     scoring=scoring,
#     n_iter=100,
#     cv=cv,
#     n_jobs=-1,
#     verbose=0,
#     random_state=42,
#     refit=True,
# ) # Run a BO search for the hyperparameters to find the ebst configuration for the dataset 

# opt.fit(X_train_np, y_train, **{'sample_weight': sample_weights}, callback=deadline)

# print("Best score:", opt.best_score_)
# print("Best params:", opt.best_params_)

In [ ]:

# model = opt.best_estimator_
# preds = model.predict(X_test_np)
# proba = model.predict_proba(X_test_np)
# print(classification_report(y_test, preds))
# auc = roc_auc_score(y_test, proba[:, 1])
# print(f"ROC-AUC: {auc:.4f}")
# joblib.dump(model,r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_lighbm_v1.plk')

In [ ]:
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, auc

# # Get positive-class probabilities
# proba = model.predict_proba(X_test_np)[:, 1]

# # Compute ROC points and AUC
# fpr, tpr, thresholds = roc_curve(y_test, proba)
# roc_auc = auc(fpr, tpr)

# # Plot
# plt.figure(figsize=(7, 6))
# plt.plot(fpr, tpr, color='darkorange', lw=2,
#          label=f'LightGBM (AUC = {roc_auc:.4f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')

# # Axis formatting
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.legend(loc='lower right')
# plt.grid(alpha=0.3)
# plt.show()

In [ ]:

# ── Model ─────────────────────────────────────────────────────────────────────

num_cont_per_step = len(num_lag_cols) // N   # 16
num_cat_per_step  = len(cat_lag_cols) // N   # 12


class CastedLinear(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_dim, in_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        return F.linear(x, self.weight.to(x.dtype))


class MLP(nn.Module):
    def __init__(self, dim, mlp_mult):
        super().__init__()
        hidden = dim * mlp_mult
        self.gate = CastedLinear(dim, hidden)
        self.up   = CastedLinear(dim, hidden)
        self.proj = CastedLinear(hidden, dim)

    def forward(self, x):
        return self.proj(F.relu(self.gate(x)) * self.up(x))


class MLPClassifier(nn.Module):
    def __init__(self, num_cont, cat_cardinalities, dim=64, mlp_mult=4, depth=2, emb_dim=8):
        super().__init__()
        self.num_cont = num_cont
        self.num_cat = len(cat_cardinalities) 
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, emb_dim) for card in cat_cardinalities
        ])
        in_dim = num_cont * N + emb_dim * len(cat_cardinalities)
        self.input_norm = nn.LayerNorm(in_dim)
        self.input_proj = CastedLinear(in_dim, dim)
        self.blocks = nn.ModuleList([MLP(dim, mlp_mult) for _ in range(depth)])
        self.head = nn.Linear(dim, 2)
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(depth)])

    def forward(self, x):
        x_cont = x[:, :self.num_cont * N].float()
        x_cat  = x[:, self.num_cont * N:].long().clamp(min=0)
        x_cat = x_cat.view(x.shape[0], N, self.num_cat)  # (B, N, 12)
        embs  = [e(x_cat[:, :, i].clamp(0, e.num_embeddings-1)).mean(dim=1)
                for i, e in enumerate(self.embeddings)]   # mean over timesteps → (B, emb_dim)
        x = torch.cat([x_cont] + embs, dim=-1)
        x = self.input_norm(x)
        x = self.input_proj(x)
        for block, norm in zip(self.blocks, self.norms):
            x = x + block(norm(x))
        return self.head(x).squeeze(-1)

In [ ]:
# # ── Model ──────────────────────────────────────────────────────────────────
# FixedMLPClassifier = partial(
#     MLPClassifier,
#     num_cont=num_cont_per_step,
#     cat_cardinalities=cat_cardinalities,
# )

# # ── Class weighting callback ─────────────────────────────────────────────────
# class DynamicClassWeights(Callback):
#     def on_train_begin(self, net, X=None, y=None, **kwargs):
#         classes = np.unique(y)
#         weights = compute_class_weight('balanced', classes=classes, y=y)
#         net.criterion_.weight = torch.tensor(weights, dtype=torch.float32)

# # ── Net ────────────────────────────────────────────────────────────────────
# net = skorch.NeuralNetClassifier(
#     module=FixedMLPClassifier,
#     module__dim=128,
#     module__mlp_mult=2,
#     module__depth=1,
#     module__emb_dim=8,
#     max_epochs=40,
#     lr=5e-4,
#     batch_size=1024,
#     iterator_train__shuffle=True,
#     train_split=skorch.dataset.ValidSplit(0.2, stratified=True),
#     verbose=0,
#     callbacks=[
#         DynamicClassWeights(),
#         GradientNormClipping(gradient_clip_value=1.0),
#         EarlyStopping(monitor='valid_loss', patience=5, lower_is_better=True),
#     ],
#     criterion=nn.CrossEntropyLoss,
# )

# # ── Search space ──────────────────────────────────────────────────────────────
# search_spaces = {
#     'lr':              Real(1e-4, 1e-3, prior='log-uniform'),
#     'module__dim':     Categorical([64, 128, 256]),
#     'module__emb_dim': Categorical([8, 16]),
# }

# scoring = make_scorer(average_precision_score,
#                        response_method="predict_proba",
#                        average='macro')

# # ── Single held-out split for CV ──────────────────────────────────────────────
# idx = np.arange(len(y_train))
# train_idx, val_idx = sklearn.model_selection.train_test_split(
#     idx, test_size=0.1, stratify=y_train, random_state=42
# )

# deadline = DeadlineStopper(total_time=60 * 60 * 3)

# opt = BayesSearchCV(
#     estimator=net,
#     search_spaces=search_spaces,
#     scoring=scoring,
#     n_iter=12,
#     cv=[(train_idx, val_idx)],
#     n_jobs=1,
#     verbose=2,
#     random_state=42,
#     refit=True,
# )

# from sklearn import set_config
# set_config(assume_finite=True)

# opt.fit(X_train_np, y_train, callback=deadline)

# print("Best score:", opt.best_score_)
# print("Best params:", opt.best_params_)

In [ ]:
# import numpy as np
# import torch
# import torch.nn as nn
# import skorch
# from skorch.dataset import ValidSplit
# from skorch.callbacks import Callback, EarlyStopping, GradientNormClipping
# from sklearn.utils.class_weight import compute_class_weight
# from functools import partial

# # ── Model ──────────────────────────────────────────────────────────────────
# FixedMLPClassifier = partial(
#     MLPClassifier,
#     num_cont=num_cont_per_step,
#     cat_cardinalities=cat_cardinalities,
# )

# # ── Class weighting callback ─────────────────────────────────────────────────
# class DynamicClassWeights(Callback):
#     def on_train_begin(self, net, X=None, y=None, **kwargs):
#         classes = np.unique(y)
#         weights = compute_class_weight('balanced', classes=classes, y=y)
#         net.criterion_.weight = torch.tensor(weights*0.5, dtype=torch.float32)

# # ── Net ────────────────────────────────────────────────────────────────────
# net = skorch.NeuralNetClassifier(
#     module=FixedMLPClassifier,
#     module__dim=128,
#     module__mlp_mult=2,
#     module__depth=1,
#     module__emb_dim=8,
#     max_epochs=50,
#     lr=9e-4,
#     batch_size=1024,
#     iterator_train__shuffle=True,
#     train_split=ValidSplit(0.2, stratified=True),
#     verbose=1,
#     callbacks=[
#         DynamicClassWeights(),
#         GradientNormClipping(gradient_clip_value=1.0),
#         EarlyStopping(monitor='valid_loss', patience=3, lower_is_better=True),
#     ],
#     criterion=nn.CrossEntropyLoss,
# )

# # ── Sanity check (optional) ──────────────────────────────────────────────────
# net.initialize()
# with torch.no_grad():
#     out = net.module_(torch.tensor(X_train_np[:32]))
#     print(out, torch.isnan(out).any())

# # ── Fit ────────────────────────────────────────────────────────────────────
# net.fit(X_train_np, y_train)

In [ ]:

# model = net
# preds = model.predict(X_test_np)
# proba = model.predict_proba(X_test_np)

# probs = net.predict_proba(X_test_np)[:, 1]
# precision, recall, thresholds = precision_recall_curve(y_test, probs)
# target_precision = 0.75
# idx = np.argmax(precision >= target_precision)
# best_thresh = thresholds[idx]
# print(best_thresh, precision[idx], recall[idx])

# # apply it
# y_pred = (probs >= best_thresh).astype(int)
# print(classification_report(y_test, y_pred))
# auc = roc_auc_score(y_test, proba[:, 1])
# print(f"ROC-AUC: {auc:.4f}")
# joblib.dump(model, r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_MLP_v1.plk')

In [ ]:
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, auc

# # Get positive-class probabilities
# proba = model.predict_proba(X_test_np)[:, 1]

# # Compute ROC points and AUC
# fpr, tpr, thresholds = roc_curve(y_test, proba)
# roc_auc = auc(fpr, tpr)

# # Plot
# plt.figure(figsize=(7, 6))
# plt.plot(fpr, tpr, color='darkorange', lw=2,
#          label=f'MLP (AUC = {roc_auc:.4f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')

# # Axis formatting
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.legend(loc='lower right')
# plt.grid(alpha=0.3)
# plt.show()

In [ ]:
# ── Retrain all models with their optimal (BO-found) hyperparameters ────────
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

results = {}

# ── Random Forest ─────────────────────────────────────────────────────────
rf_params = OrderedDict([('bootstrap', True), ('class_weight', 'balanced_subsample'),
                          ('max_depth', 28), ('max_features', 'sqrt'),
                          ('min_samples_leaf', 7), ('n_estimators', 858)])
rf_model = RandomForestClassifier(random_state=42, **rf_params)
rf_model.fit(X_train_np, y_train, sample_weight=sample_weights)
preds = rf_model.predict(X_test_np)
proba = rf_model.predict_proba(X_test_np)
print("=== Random Forest ===")
print(classification_report(y_test, preds))
print(f"ROC-AUC: {roc_auc_score(y_test, proba[:, 1]):.4f}")
joblib.dump(rf_model, r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_RF_v1.plk')
results['Random Forest'] = proba[:, 1]

# ── XGBoost ──────────────────────────────────────────────────────────────
xgb_params = OrderedDict([('colsample_bytree', 1.0), ('learning_rate', 0.2034370190697667),
                           ('max_delta_step', 3), ('max_depth', 10), ('min_child_weight', 5),
                           ('n_estimators', 2000), ('reg_alpha', 1e-09), ('reg_lambda', 1e-09),
                           ('subsample', 0.8371868846738777)])
xgb_model = xgb.XGBClassifier(enable_categorical=True, tree_method='hist',
                               eval_metric='aucpr', random_state=42, **xgb_params)
xgb_model.fit(X_train_np, y_train, sample_weight=sample_weights)
preds = xgb_model.predict(X_test_np)
proba = xgb_model.predict_proba(X_test_np)
print("=== XGBoost ===")
print(classification_report(y_test, preds))
print(f"ROC-AUC: {roc_auc_score(y_test, proba[:, 1]):.4f}")
joblib.dump(xgb_model, r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_xgb_v1.plk')
results['XGBoost'] = proba[:, 1]

# ── LightGBM ─────────────────────────────────────────────────────────────
lgbm_params = OrderedDict([('colsample_bytree', 0.5450117686859245), ('learning_rate', 0.21661693127982293),
                            ('max_depth', 12), ('min_child_samples', 74), ('n_estimators', 1000),
                            ('num_leaves', 1000), ('reg_alpha', 1.1515833248696066),
                            ('reg_lambda', 0.20493132534038447), ('subsample', 0.7598010881577231)])
lgbm_model = gbm.LGBMClassifier(objective='binary', random_state=42, n_jobs=-1, **lgbm_params)
lgbm_model.fit(X_train_np, y_train, sample_weight=sample_weights)
preds = lgbm_model.predict(X_test_np)
proba = lgbm_model.predict_proba(X_test_np)
print("=== LightGBM ===")
print(classification_report(y_test, preds))
print(f"ROC-AUC: {roc_auc_score(y_test, proba[:, 1]):.4f}")
joblib.dump(lgbm_model, r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_lighbm_v1.plk')
results['LightGBM'] = proba[:, 1]

# ── Logistic Regression ─────────────────────────────────────────────────────
# Continuous block is already scaled by the preprocessing pipeline; the
# categorical block holds integer codes, so one-hot encode those for a
# linear model (tree/boosting models above consume the codes directly).
x_cont_train, x_cat_train = X_train_np[:, :num_cont_per_step * N], X_train_np[:, num_cont_per_step * N:]
x_cont_test,  x_cat_test  = X_test_np[:, :num_cont_per_step * N],  X_test_np[:, num_cont_per_step * N:]

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
x_cat_train_oh = ohe.fit_transform(x_cat_train)
x_cat_test_oh = ohe.transform(x_cat_test)

X_train_lr = sp.hstack([sp.csr_matrix(x_cont_train), x_cat_train_oh]).tocsr()
X_test_lr = sp.hstack([sp.csr_matrix(x_cont_test), x_cat_test_oh]).tocsr()

lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_lr, y_train)
preds = lr_model.predict(X_test_lr)
proba = lr_model.predict_proba(X_test_lr)
print("=== Logistic Regression ===")
print(classification_report(y_test, preds))
print(f"ROC-AUC: {roc_auc_score(y_test, proba[:, 1]):.4f}")
joblib.dump(lr_model, r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_LR_v1.plk')
results['Logistic Regression'] = proba[:, 1]

# ── Linear model with categorical embeddings ────────────────────────────────
# Reuses MLPClassifier's embedding + input-projection front end but with
# depth=0 (no MLP blocks between the projection and the output head), so the
# network is a linear map on top of the learned categorical embeddings.
FixedLinearEmbClassifier = partial(
    MLPClassifier,
    num_cont=num_cont_per_step,
    cat_cardinalities=cat_cardinalities,
    depth=0,
)

class DynamicClassWeightsLinEmb(Callback):
    def on_train_begin(self, net, X=None, y=None, **kwargs):
        classes = np.unique(y)
        weights = compute_class_weight('balanced', classes=classes, y=y)
        net.criterion_.weight = torch.tensor(weights, dtype=torch.float32)

linear_emb_net = skorch.NeuralNetClassifier(
    module=FixedLinearEmbClassifier,
    module__dim=128,
    module__mlp_mult=1,
    module__emb_dim=8,
    max_epochs=50,
    lr=9e-4,
    batch_size=1024,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.2, stratified=True),
    verbose=1,
    callbacks=[
        DynamicClassWeightsLinEmb(),
        GradientNormClipping(gradient_clip_value=1.0),
        EarlyStopping(monitor='valid_loss', patience=3, lower_is_better=True),
    ],
    criterion=nn.CrossEntropyLoss,
)
linear_emb_net.fit(X_train_np, y_train)
preds = linear_emb_net.predict(X_test_np)
proba = linear_emb_net.predict_proba(X_test_np)
print("=== Linear + Embeddings ===")
print(classification_report(y_test, preds))
print(f"ROC-AUC: {roc_auc_score(y_test, proba[:, 1]):.4f}")
joblib.dump(linear_emb_net, r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_LinEmb_v1.plk')
results['Linear + Embeddings'] = proba[:, 1]

# ── MLP (skorch) ─────────────────────────────────────────────────────────
FixedMLPClassifier = partial(
    MLPClassifier,
    num_cont=num_cont_per_step,
    cat_cardinalities=cat_cardinalities,
)

class DynamicClassWeights(Callback):
    def on_train_begin(self, net, X=None, y=None, **kwargs):
        classes = np.unique(y)
        weights = compute_class_weight('balanced', classes=classes, y=y)
        net.criterion_.weight = torch.tensor(weights * 0.5, dtype=torch.float32)

net = skorch.NeuralNetClassifier(
    module=FixedMLPClassifier,
    module__dim=128,
    module__mlp_mult=2,
    module__depth=1,
    module__emb_dim=8,
    max_epochs=50,
    lr=9e-4,
    batch_size=1024,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.2, stratified=True),
    verbose=1,
    callbacks=[
        DynamicClassWeights(),
        GradientNormClipping(gradient_clip_value=1.0),
        EarlyStopping(monitor='valid_loss', patience=3, lower_is_better=True),
    ],
    criterion=nn.CrossEntropyLoss,
)
net.fit(X_train_np, y_train)
preds = net.predict(X_test_np)
proba = net.predict_proba(X_test_np)
print("=== MLP ===")
print(classification_report(y_test, preds))
print(f"ROC-AUC: {roc_auc_score(y_test, proba[:, 1]):.4f}")
joblib.dump(net, r'/Users/cbrou/Documents/figw_memoire_IBM/models_H1\model_MLP_v1.plk')
results['MLP'] = proba[:, 1]

# ── Combined ROC-AUC plot ────────────────────────────────────────────────
plt.figure(figsize=(7, 6))
colors = {'Random Forest': 'darkorange', 'XGBoost': 'seagreen',
          'LightGBM': 'royalblue', 'Logistic Regression': 'purple',
          'Linear + Embeddings': 'goldenrod', 'MLP': 'firebrick'}
for name, proba_pos in results.items():
    fpr, tpr, _ = roc_curve(y_test, proba_pos)
    roc_auc = sk_auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors.get(name), lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — All Models')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()
